**Hands-on Lab: Interactive Visual Analytics with Folium**

In [1]:
import pandas as pd 
import folium

from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

In [2]:
url="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df=pd.read_csv(url)
spacex_df.sample(5)

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
18,20,2015-12-22,1:29:00,F9 FT B1019,CCAFS LC-40,OG2 Mission 2 11 Orbcomm-OG2 satellites,2034.0,LEO,Orbcomm,Success (ground pad),1,28.562302,-80.577356
42,36,2017-06-23,19:10:00,F9 FT B1029.2,KSC LC-39A,BulgariaSat-1,3669.0,GTO,Bulsatcom,Success (drone ship),1,28.573255,-80.646895
24,27,2016-07-18,4:45:00,F9 FT B1025.1,CCAFS LC-40,SpaceX CRS-9,2257.0,LEO (ISS),NASA (CRS),Success (ground pad),1,28.562302,-80.577356
30,40,2017-08-24,18:51:00,F9 FT B1038.1,VAFB SLC-4E,Formosat-5,475.0,SSO,NSPO,Success (drone ship),1,34.632834,-120.610745
36,30,2017-02-19,14:39:00,F9 FT B1031.1,KSC LC-39A,SpaceX CRS-10,2490.0,LEO (ISS),NASA (CRS),Success (ground pad),1,28.573255,-80.646895


In [3]:
spacex_df=spacex_df[["Launch Site","Lat","Long", "class"]]
launch_sites_df=spacex_df.groupby(["Launch Site"], as_index=False).first()
launch_sites_df=launch_sites_df[["Launch Site","Lat","Long"]]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [4]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map=folium.Map(location=nasa_coordinate, zoom_start=10)
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)
site_map

In [5]:
for ls, lat, long in zip(launch_sites_df["Launch Site"], launch_sites_df["Lat"], launch_sites_df["Long"]):
    coordinate=[lat,long]
    folium.Marker(location=coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % ls, )).add_to(site_map)
    folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(ls)).add_to(site_map)

site_map

In [6]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [7]:
marker_cluster=MarkerCluster().add_to(site_map)
colores=[]
for clase in spacex_df["class"]:
    if clase==1:
        colores.append("green")
    else:
        colores.append("red")

spacex_df["marker_color"]=colores

spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [8]:
for index, record in spacex_df.iterrows():
    coords=record["Lat"], record["Long"]
    color=record["marker_color"]
    marker=folium.Marker(location=coords, color=color)
    marker_cluster.add_child(marker)

site_map

In [9]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [10]:
point_coordinate = [28.61406, -80.67827]

In [11]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [12]:
punto_dos=spacex_df.loc[0,["Lat","Long"]]
distance_coastline=calculate_distance(point_coordinate[0],point_coordinate[1],punto_dos[0],punto_dos[1])
distance_coastline

C:\Users\leona\AppData\Local\Temp\ipykernel_28448\4032450882.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  distance_coastline=calculate_distance(point_coordinate[0],point_coordinate[1],punto_dos[0],punto_dos[1])


11.414305649437612

In [13]:
folium.Marker(location=point_coordinate, icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )).add_to(site_map)

site_map

In [ ]:
lines=folium.PolyLine(locationss=[[point_coordinate[0],point_coordinate[1]],[punto_dos[0],punto_dos[1]]], weight=1)
site_map.add_child(lines)

C:\Users\leona\AppData\Local\Temp\ipykernel_28448\3677520336.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lines=folium.PolyLine(locations=[[point_coordinate[0],point_coordinate[1]],[punto_dos[0],punto_dos[1]]], weight=1)
